# PHƯƠNG ÁN B — MI–RF-RFE step=2–SVM coarse-to-fine

Dataset duy nhất: **MyDrive/CICFLOWMETER_FEATURE_CSV**. Namespace mới: **MyDrive/experiment_results/phuong_anB_step2/mi_rfe_svm/**. Chỉ feature selection trên Train/Validation; không final refit và không đọc Test. **EXPERIMENTAL_NON_OFFICIAL / OFFICIAL_BLOCKED**.

**Dùng đúng policy run_phuong_anB.py do người dùng cung cấp:** coarse=(10,20,30,50,75,100); chọn coarse best theo Validation Accuracy, hòa k nhỏ nhất; fine xét các k chưa thử trong [max(1,best−5),min(100,best+5)] theo thứ tự tăng dần. Sau fine, chọn best theo cùng tiêu chí. Không còn các policy tự đề xuất hoặc hộp thoại chọn policy mới.

**RFE_STEP=2 chỉ điều khiển RF elimination, không phải khoảng cách coarse grid.** So với B gốc step1, chỉ đổi RF-RFE sang step2; MI/scaler/SVM giữ helper A1 đã khóa hash. Step2 có thể đổi ranking, được bind riêng trong identity; không nhập ranking step1/checkpoint A1 hoặc B bản nháp.

Mỗi config/run tách biệt. Mặc định chọn một config; có thể chọn cả bốn để queue tuần tự. RFE Top100 step2 có 50 vòng loại bỏ + 1 RF fit cuối = 51 RF fit; coarse có 6 SVM fit, fine 5 hoặc 10 SVM fit theo anchor. Đây là số fit theo lịch, không phải benchmark thời gian hoặc cam kết Accuracy.

Chạy Cell 1–8 để setup/test/preflight/xác nhận; Cell 9 mặc định EXECUTE_QUEUE=False. Cell 10 monitor, Cell 11 safe stop, Cell 12 chuẩn bị resume, Cell 13 kết quả, Cell 14 verify. Notebook chưa được Codex thực thi.


## Cell 1 — Cài dependencies khi bạn chủ động

PREPARE_ENVIRONMENT=False mặc định. Nếu cần cài, bật True, chạy rồi restart session; đặt về False trước khi tiếp tục. Không đổi thư viện khi worker còn chạy. Giữ Python/platform phù hợp profile; không sửa manifest để bỏ mismatch.


In [ ]:
PREPARE_ENVIRONMENT = False
if PREPARE_ENVIRONMENT:
    if globals().get('JOB', {}).get('running'):
        raise RuntimeError('Worker đang chạy; không đổi môi trường.')
    import subprocess
    import sys
    if not (3, 10) <= sys.version_info[:2] < (3, 14):
        raise RuntimeError('Profile hỗ trợ Python 3.10..3.13.')
    numpy_pin = '1.26.4' if sys.version_info[:2] < (3, 13) else '2.2.6'
    subprocess.run([sys.executable, '-m', 'pip', 'install', '--no-cache-dir',
        'numpy==' + numpy_pin, 'pandas==2.3.3', 'scipy==1.15.3',
        'scikit-learn==1.7.2', 'joblib==1.5.3', 'psutil==5.9.5',
        'pyarrow==23.0.1', 'tqdm==4.67.1'], check=True)
    raise RuntimeError('Cài xong. Restart session; đặt PREPARE_ENVIRONMENT=False rồi chạy lại.')
else:
    print('Không cài hoặc thay đổi thư viện.')


## Cell 2 — Mount Google Drive

Nếu credential propagation thất bại, dừng tại đây, hoàn tất xác thực Google trong Colab rồi chạy lại cell. Không force remount tự động và không tạo output giả ở đường dẫn chưa mount.


In [ ]:
if globals().get('JOB', {}).get('running'):
    raise RuntimeError('Không mount lại khi queue đang chạy.')
from pathlib import Path
from google.colab import drive
try:
    drive.mount('/content/drive')
except Exception as exc:
    raise RuntimeError('Mount Drive thất bại. Hoàn tất xác thực Google/Colab rồi chạy lại; chưa training.') from exc
DRIVE = Path('/content/drive/MyDrive')
if not DRIVE.is_dir():
    raise RuntimeError('MyDrive chưa truy cập được; dừng setup.')
print('DRIVE:', DRIVE)


## Cell 3 — Dataset, config, run và policy gốc

APPROVED_COMMIT là SHA mới chứa source B đã sửa. COARSE và RADIUS được khóa theo run_phuong_anB.py; không chỉnh để né preflight. Chọn một config để chạy từng cấu hình hoặc list(CONFIGS) để chạy tuần tự. Giữ đúng Run ID khi resume; không dùng ID A1.


In [ ]:
if globals().get('JOB', {}).get('running'):
    raise RuntimeError('Queue đang chạy; không thay cấu hình.')
import json, os, re, subprocess, sys
DATASET_ROOT = DRIVE / 'CICFLOWMETER_FEATURE_CSV'
OUTPUT_ROOT = DRIVE / 'experiment_results'
LOCAL_ROOT = Path('/content/mi_rfe_svm_b_step2')
CONFIGS = ['FW256_FS8', 'FW256_FS16', 'FW512_FS8', 'FW512_FS16']
SELECTED_CONFIGS = ['FW256_FS8']  # hoặc list(CONFIGS) để chạy tuần tự đủ bốn
RUN_ID = 'RUN_B2_01'
RUN_IDS = {config: RUN_ID for config in CONFIGS}
RFE_STEP = 2
APPROVED_COMMIT = ''  # SHA40 sau khi bạn push source B + notebook mới
SEARCH_POLICY = 'phuong_anB_coarse_to_fine_v1'
COARSE = (10, 20, 30, 50, 75, 100)
RADIUS = 5  # độc lập với RFE_STEP=2
EVIDENCE_ROOTS = {}  # config -> original frozen metadata root; mặc định resolver kiểm tra SHA snapshot
TAKEOVER = {}  # chỉ điền PREVIOUS RUNTIME STOPPED <ID> khi worker cũ đã thực sự dừng
ACKNOWLEDGE_LOCAL_STOP = False
RUNTIME = {'session_budget_seconds': 36000, 'stop_margin_seconds': 1800, 'heartbeat_seconds': 15}
BUDGET = {'min_available_ram_bytes': 2*1024**3, 'max_process_rss_bytes': 40*1024**3}
if not SELECTED_CONFIGS or len(set(SELECTED_CONFIGS)) != len(SELECTED_CONFIGS) or not set(SELECTED_CONFIGS) <= set(CONFIGS):
    raise ValueError('Chọn config trong bốn cấu hình CICFLOW, không trùng.')
if RFE_STEP != 2:
    raise ValueError('Notebook B được khóa step=2.')
if not DATASET_ROOT.is_dir():
    raise FileNotFoundError(DATASET_ROOT)
for config in SELECTED_CONFIGS:
    print(config, RUN_IDS[config], '->', OUTPUT_ROOT / 'phuong_anB_step2/mi_rfe_svm' / config / RUN_IDS[config])
print('POLICY:', SEARCH_POLICY, '| chưa training')


## Cell 4 — Checkout B riêng và kiểm tra môi trường

Không pull/reset checkout đang dùng. Nếu source thiếu B hoặc kernel đang giữ module từ checkout khác, dừng; không monkeypatch source để bỏ identity gate. Source B và A1 dùng các checkout độc lập.


In [ ]:
if globals().get('JOB', {}).get('running'):
    raise RuntimeError('Worker đang chạy; không đổi source.')
if not re.fullmatch(r'[0-9a-fA-F]{40}', APPROVED_COMMIT):
    raise ValueError('Điền SHA40 commit mới có phuong_anB_step2 ở Cell 3.')
APPROVED_COMMIT = APPROVED_COMMIT.lower()
REPO = Path('/content/CICFLOW_B_STEP2_' + APPROVED_COMMIT[:12])
if not REPO.exists():
    subprocess.run(['git', 'clone', '--', 'https://github.com/kasum00/RFforNetwork.git', str(REPO)], check=True)
    subprocess.run(['git', '-C', str(REPO), 'checkout', '--detach', APPROVED_COMMIT], check=True)
def git_read(*args):
    return subprocess.run(['git', '-C', str(REPO), *args], check=True,
                          capture_output=True, text=True).stdout.strip()
if git_read('rev-parse', 'HEAD') != APPROVED_COMMIT or git_read('status', '--porcelain'):
    raise RuntimeError('Checkout không sạch/không đúng SHA; không tự reset.')
SOURCE_ROOT = REPO / 'code_CICFLOW'
if not (SOURCE_ROOT / 'phuong_anB_step2/pipeline.py').is_file():
    raise RuntimeError('Commit chưa có source B. Push đúng package B + notebook mới rồi dùng SHA mới.')
sys.dont_write_bytecode = True
for name, module in list(sys.modules.items()):
    if name == 'phuong_anB_step2' or name.startswith('phuong_anB_step2.') or name == 'mi_rfe_svm_colab' or name.startswith('mi_rfe_svm_colab.'):
        file = getattr(module, '__file__', None)
        if file and SOURCE_ROOT.resolve() not in Path(file).resolve().parents:
            raise RuntimeError('Kernel giữ module từ checkout khác. Chờ worker cũ dừng rồi dùng session sạch.')
if str(SOURCE_ROOT) not in sys.path:
    sys.path.insert(0, str(SOURCE_ROOT))
from phuong_anB_step2.environment import environment
ENVIRONMENT = environment()
print(json.dumps(ENVIRONMENT, indent=2))
if ENVIRONMENT['errors']:
    raise RuntimeError('Sai dependencies; dùng Cell 1 rồi restart, không sửa gate.')
from phuong_anB_step2.contract import paths, read, confirmation, digest
from phuong_anB_step2.controller import launch, launch_tests
from phuong_anB_step2.policy import policy_spec, approved_policy, POLICY_NAME
from phuong_anB_step2.notebook_helpers import build_plan, planned_action, Queue, monitor, summarize
TEST_EVIDENCE = {'status': 'NOT_RUN'}
PLAN, CONFIRMED_REQUESTS = [], []
QUEUE, JOB = None, {'running': False, 'error': None}
print('B SOURCE:', SOURCE_ROOT, '| COMMIT:', APPROVED_COMMIT)


## Cell 5 — Đúng dataset A1: kiểm tra manifest nhỏ

So sánh nguyên bytes/hash sidecar với snapshot dataset đã đọc khi triển khai. Chưa đọc Train/Validation values hoặc Test; đây chưa phải full preflight PASS. Parent evidence, full Train/Validation SHA và schema/footer sẽ được worker kiểm tra ở Cell 7.


In [ ]:
if globals().get('JOB', {}).get('running'):
    raise RuntimeError('Chờ queue dừng trước khi kiểm kê lại.')
from phuong_anB_step2.dataset import discover, validate_dataset_reference
DATASET_REFERENCE = read(SOURCE_ROOT / 'phuong_anB_step2/evidence/dataset_reference.json')
CATALOG = discover(DATASET_ROOT)
print('DATASET:', DATASET_ROOT)
print('CATALOG:', json.dumps(CATALOG, indent=2))
if not set(SELECTED_CONFIGS) <= set(CATALOG['configs']):
    raise RuntimeError('Có config thiếu manifest hoặc Stage3 chưa complete.')
for config in CONFIGS:
    expected = DATASET_REFERENCE['configs'][config]
    validate_dataset_reference(DATASET_ROOT / config, expected)
    print(config, 'frozen sidecars matched | windows:', expected['windows_by_split'],
          '| features:', expected['feature_count'], '| label:', expected['label'])
print('Metadata matched. Chưa fit; Test chưa mở; chưa kết luận full preflight PASS.')


## Cell 6 — Unit tests synthetic

Người dùng chủ động chạy; fake SVC/RF và fixture nhỏ, không đọc dataset production. Evidence phải khớp source/environment trước khi training. Tất cả output và native exit được ghi log tổng hợp; không gắn PASS chỉ từ exit=0.


In [ ]:
RUN_SYNTHETIC_TESTS = True
if globals().get('JOB', {}).get('running'):
    raise RuntimeError('Không chạy tests khi queue đang chạy.')
if RUN_SYNTHETIC_TESTS:
    TEST_TRANSPORT, TEST_EVIDENCE = launch_tests(SOURCE_ROOT, LOCAL_ROOT, DRIVE)
    print(json.dumps(TEST_EVIDENCE, indent=2))
    if TEST_TRANSPORT['native_exit_code'] != 0 or TEST_EVIDENCE['status'] != 'PASS':
        raise RuntimeError('B tests chưa PASS; không preflight/training.')
else:
    print('Không tạo evidence mới; preflight chặn nếu evidence thiếu/stale.')


## Cell 7 — Preflight và queue plan theo policy gốc

In rõ coarse sáu mức, radius=5 và RFE step=2. Không yêu cầu chọn policy thay thế: source người dùng đã chốt chính sách. Preflight kiểm tra đủ source/Git, thư viện, dataset/frozen evidence, policy/version, resource và checkpoint. Không training ở cell này.


In [ ]:
if globals().get('JOB', {}).get('running'):
    raise RuntimeError('Queue đang chạy; không tạo plan mới.')
PLAN, CONFIRMED_REQUESTS = [], []
POLICY = approved_policy({'rfe_step': RFE_STEP, 'search_policy': SEARCH_POLICY,
                          'coarse': list(COARSE), 'radius': RADIUS})
print('ORIGINAL SEARCH POLICY:', json.dumps(POLICY, indent=2))
print('RF-RFE step:', RFE_STEP, '| Coarse SVM k:', COARSE, '| Fine radius:', RADIUS)

def request_for(config, mode='preflight'):
    return {'source': str(SOURCE_ROOT), 'git_commit': APPROVED_COMMIT,
        'dataset': str(DATASET_ROOT), 'output': str(OUTPUT_ROOT), 'local_root': str(LOCAL_ROOT),
        'config': config, 'run_id': RUN_IDS[config], 'mode': mode, 'rfe_step': RFE_STEP,
        'search_policy': SEARCH_POLICY, 'coarse': list(COARSE), 'radius': RADIUS,
        'runtime': dict(RUNTIME), 'budget': dict(BUDGET), 'unit_tests': TEST_EVIDENCE,
        'evidence_root': EVIDENCE_ROOTS.get(config),
        'takeover_confirmation': TAKEOVER.get(config),
        'acknowledge_local_stop': ACKNOWLEDGE_LOCAL_STOP}

CURRENT_REQUESTS = [request_for(config) for config in SELECTED_CONFIGS]
PLAN = build_plan(CURRENT_REQUESTS)
PLAN_SIGNATURE = digest([item['request'] for item in PLAN])
print('\nQUEUE PLAN')
for item in PLAN:
    req = item['request']
    print(req['config'], req['run_id'], req['mode'], 'READY' if item['ready'] else 'BLOCKED')
if not PLAN or not all(item['ready'] for item in PLAN):
    raise RuntimeError('Có config BLOCKED/FAIL. Không tự đổi ID hoặc bỏ gate. Chưa training.')
print('PREFLIGHT PASS — chưa training.')


## Cell 8 — Xác nhận từng run trước khi training

Chỉ các request có preflight READY được đưa vào batch. Chọn một config ở Cell 3 để chạy từng cấu hình. Có nhiều config thì queue tuần tự; mỗi config complete sẽ được verify trước khi sang config tiếp theo.


In [ ]:
if globals().get('JOB', {}).get('running'):
    raise RuntimeError('Queue đang chạy.')
if not PLAN or not all(item['ready'] for item in PLAN):
    raise RuntimeError('Cell 7 chưa có plan READY đầy đủ.')
CONFIRMED_REQUESTS = []
for item in PLAN:
    req = json.loads(json.dumps(item['request']))
    if req['mode'] in ('run', 'resume'):
        exact = confirmation(req)
        if input('Nhập đúng để cho phép request này:\n' + exact + '\n') != exact:
            CONFIRMED_REQUESTS = []
            raise RuntimeError('Confirmation differs; queue chưa bắt đầu.')
        req['confirmation'] = exact
    CONFIRMED_REQUESTS.append(req)
print('Đã chốt', len(CONFIRMED_REQUESTS), 'request. Chưa bắt đầu worker.')


## Cell 9 — Chạy B (mặc định tắt)

Chỉ đổi EXECUTE_QUEUE=True sau khi xem plan và xác nhận Cell 8. Cell trả quyền điều khiển khi worker còn chạy nền; dùng Cell 10. Không chạy lại setup, cài package hoặc Cell 9 lần nữa khi JOB còn running.


In [ ]:
EXECUTE_QUEUE = False
if not EXECUTE_QUEUE:
    print('Chưa training. Đặt EXECUTE_QUEUE=True rồi chạy lại Cell 9 khi đã sẵn sàng.')
else:
    if globals().get('JOB', {}).get('running'):
        raise RuntimeError('Queue vẫn chạy; không tạo worker trùng.')
    current = [{**request_for(config), 'mode': planned_action(request_for(config))} for config in SELECTED_CONFIGS]
    normalized = [{k: v for k, v in req.items() if k != 'confirmation'} for req in CONFIRMED_REQUESTS]
    if (not PLAN or not all(item['ready'] for item in PLAN)
            or len(CONFIRMED_REQUESTS) != len(PLAN)
            or digest(current) != PLAN_SIGNATURE or digest(normalized) != PLAN_SIGNATURE):
        raise RuntimeError('Plan/confirmation stale hoặc cấu hình đã đổi; chạy lại Cell 7–8.')
    QUEUE = Queue()
    JOB = QUEUE.state
    QUEUE.start(CONFIRMED_REQUESTS)
    print('B queue started. Theo dõi ở Cell 10.')


## Cell 10 — Progress bar thật + heartbeat

Bar chỉ tăng theo đơn vị đã COMMITTED; không đoán % của SVM đang fit. Hiển thị phase COARSE/FINE, k, best k/Accuracy, elapsed, ETA từ thời gian các đơn vị đã hoàn thành, CPU/RAM và checkpoint cuối. NO HEARTBEAT của config chưa chạy là trạng thái chờ. Dừng cell monitor không gửi safe stop.


In [ ]:
MONITOR_REQUESTS = CONFIRMED_REQUESTS or CURRENT_REQUESTS
monitor(MONITOR_REQUESTS, QUEUE, refresh_seconds=15, max_refresh=0)


## Cell 11 — Safe stop hoặc dừng queue sau config hiện tại

REQUEST_STOP=True gửi cờ dừng tại safe boundary và bỏ các config chưa chạy. STOP_AFTER_CONFIG=True để config hiện tại complete/verify rồi dừng queue. Không bật cả hai. Chờ PAUSED và checkpoint SAVED hoặc selection complete đã verify trước khi ngắt runtime.


In [ ]:
REQUEST_STOP = False
STOP_AFTER_CONFIG = False
if REQUEST_STOP and STOP_AFTER_CONFIG:
    raise ValueError('Chỉ chọn một kiểu dừng.')
if REQUEST_STOP or STOP_AFTER_CONFIG:
    if QUEUE is None:
        raise RuntimeError('Notebook không quản lý worker nào; không tự takeover từ heartbeat.')
    print(QUEUE.safe_stop() if REQUEST_STOP else QUEUE.stop_after_config())
else:
    print('Không gửi lệnh dừng.')


## Cell 12 — Chuẩn bị resume checkpoint

Giữ nguyên source commit, dataset/fingerprint, environment, policy, step và Run ID. Chỉ đơn vị đã commit hoặc local record được xác minh mới được dùng lại; fit SVC bị ngắt phải tính lại. Mất runtime có thể cần dựng cache Train/Validation bằng scaler đã lưu. Lease chưa release vẫn BLOCKED; chỉ điền TAKEOVER khi worker cũ thực sự đã dừng. Chưa có Test/final command.


In [ ]:
PREPARE_RESUME = False
if PREPARE_RESUME:
    if globals().get('JOB', {}).get('running'):
        raise RuntimeError('Worker còn chạy; không resume trùng.')
    ACKNOWLEDGE_LOCAL_STOP = True
    PLAN, CONFIRMED_REQUESTS = [], []
    print('Đã chuẩn bị acknowledge local stop. Không training tại Cell 12.')
    print('Kiểm tra đúng RUN_IDS/SEARCH_POLICY/commit, rồi chạy lại Cell 7 → 8 → 9.')
else:
    print('Không thay đổi trạng thái resume.')


## Cell 13 — Tổng hợp best k, Validation và thống kê

Chỉ đọc kết quả đã lưu; SAVED_REPORT_ONLY chưa phải verify SHA trong session này. Tổng thời gian selection là tổng wall receipts đã lưu đến khi hoàn thành công việc; thời gian mất khi runtime chết đột ngột không được bịa lại. Không so tốc độ với thời gian resume riêng của A1 như thể đó là toàn bộ pipeline.


In [ ]:
REPORT_REQUESTS = [request_for(config) for config in SELECTED_CONFIGS]
summarize(REPORT_REQUESTS)
for req in REPORT_REQUESTS:
    _, remote = paths(req)
    stats_path = remote / 'results/statistics.json'
    if stats_path.is_file():
        stats = read(stats_path)
        print('\n', req['config'], req['run_id'])
        for key in ('rf_fits', 'svm_fits', 'svm_coarse_fits', 'svm_fine_fits',
                    'rf_fit_attempts_recorded', 'svm_fit_attempts_recorded',
                    'mi_seconds', 'rf_rfe_fit_seconds', 'coarse_seconds', 'fine_seconds',
                    'total_selection_seconds', 'total_selection_time_scope'):
            print(key, ':', stats[key])
print('Aggregate log:', DRIVE / 'code_CICFLOW/logs/ALL_EXECUTION_RESULTS.log')


## Cell 14 — Verify artifacts hoàn chỉnh

Mặc định tắt; queue đã verify tự động sau mỗi config complete. Verify giữ nguyên gates source/policy/dataset/environment và kiểm tra SHA chain/publication, không fit hoặc đọc Test. Không coi exit=0 một mình là khoa học PASS.


In [ ]:
VERIFY_ARTIFACTS = False
if VERIFY_ARTIFACTS:
    if globals().get('JOB', {}).get('running'):
        raise RuntimeError('Chờ queue dừng trước khi verify riêng.')
    for config in SELECTED_CONFIGS:
        req = request_for(config, 'verify')
        _, remote = paths(req)
        if not (remote / 'results/SELECTION_COMPLETE.json').is_file():
            print(config, 'NOT_COMPLETE — chưa verify selection.')
            continue
        verified = launch(req)
        print(json.dumps(verified, indent=2, ensure_ascii=False))
        if verified['native_exit_code'] != 0 or verified['status'] != 'PASS' or verified.get('phase_status') != 'SELECTION_COMPLETE':
            raise RuntimeError('B verify chưa PASS; giữ nguyên artifacts và xem log.')
else:
    print('Không chạy verify mới. Không final refit, không đánh giá Test.')
